In [ ]:
import pandas as pd
import numpy as np

In [3]:
pd.set_option('display.max_columns', 50)

In [2]:
df = pd.read_csv("merged_streaming_data.csv")
df.head()

,id_student,code_module,code_presentation,date,forumng,homepage,oucontent,subpage,url,resource,...,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,date_registration,date_unregistration
0,6516,AAA,2014J,-23.0,0,3,23,2,0,0,...,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN
1,6516,AAA,2014J,-22.0,33,13,34,0,0,2,...,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN
2,6516,AAA,2014J,-20.0,13,12,8,1,0,7,...,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN
3,6516,AAA,2014J,-17.0,0,2,0,3,2,0,...,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN
4,6516,AAA,2014J,-12.0,1,1,0,0,0,0,...,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN


In [4]:
df.columns

Index(['id_student', 'code_module', 'code_presentation', 'date', 'forumng',
       'homepage', 'oucontent', 'subpage', 'url', 'resource', 'glossary',
       'dataplus', 'oucollaborate', 'quiz', 'ouelluminate', 'sharedsubpage',
       'questionnaire', 'page', 'externalquiz', 'ouwiki', 'dualpane',
       'repeatactivity', 'folder', 'htmlactivity', 'id_assessment',
       'is_banked', 'score', 'assessment_type', 'due_date', 'weight',
       'module_presentation_length', 'gender', 'region', 'highest_education',
       'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits',
       'disability', 'final_result', 'date_registration',
       'date_unregistration'],
      dtype='object')

In [8]:
course_student_cols = df.columns[:3].tolist() + df.columns[30:].tolist()
course_student_vle_cols = df.columns[:24].tolist() + df.columns[30:].tolist()
course_student_assessment_cols = df.columns[:4].tolist() + df.columns[24:].tolist()

Subset df according to the columns above depending on the analyses you want to do

In [ ]:
# e.g., if I'm only interested in learning about the demographics of students in each course: 
df_demo = df[course_student_cols].drop_duplicates()
df_demo.head()

,id_student,code_module,code_presentation,module_presentation_length,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,date_registration,date_unregistration
0,6516,AAA,2014J,269,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN
159,8462,DDD,2013J,261,M,London Region,HE Qualification,30-40%,55<=,0,90,N,Withdrawn,-137.0,119.0
215,8462,DDD,2014J,262,M,London Region,HE Qualification,30-40%,55<=,1,60,N,Withdrawn,-38.0,18.0
220,11391,AAA,2013J,268,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,-159.0,NaN
263,23629,BBB,2013B,240,F,East Anglian Region,Lower Than A Level,20-30%,0-35,2,60,N,Fail,-47.0,NaN


In [ ]:
# e.g., if I'm only interested in vle data of students in each course: 
df_vle = df[course_student_vle_cols].drop_duplicates()
df_vle.head()

# there should be no 'date' duplicated for each (id_student,code_module,code_presentation) combination
# check: 
# has_duplicates = df_demo.duplicated(
#     subset=["id_student", "code_module", "code_presentation", "date"]
# ).any()
# print(has_duplicates) # should be False


False


In [ ]:
# e.g., if I'm only interested in assessment data of students in each course: 
df_assessment = df[course_student_assessment_cols].dropna(subset=["id_assessment"])
df_assessment.head()

# NB: There could be duplicate date for each (id_student,code_module,code_presentation) combination if they submitted more than one assessment on a particular date

,id_student,code_module,code_presentation,date,id_assessment,is_banked,score,assessment_type,due_date,weight,module_presentation_length,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,date_registration,date_unregistration
21,6516,AAA,2014J,17.0,1758.0,0.0,60.0,TMA,19.0,10.0,269,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN
40,6516,AAA,2014J,51.0,1759.0,0.0,48.0,TMA,54.0,20.0,269,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN
73,6516,AAA,2014J,116.0,1760.0,0.0,63.0,TMA,117.0,20.0,269,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN
103,6516,AAA,2014J,164.0,1761.0,0.0,61.0,TMA,166.0,20.0,269,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN
136,6516,AAA,2014J,210.0,1762.0,0.0,77.0,TMA,215.0,30.0,269,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN


If I want to combine both vle and assessments, then some sort of summarizing would be required to deal with the duplicate rows in the original df. For example, if looking at metrics by week, it might be easier to calculate those weekly summarize for df_vle and df_assessment separately, and then combine them again based on a similar columns (e.g., ['id_student','code_module',code_prsentation','week']). 

In [ ]:
# Example code for merging two dataframes that summarized df_vle by week and df_assessment by week

common_columns = list(set(df_vle_byweek.columns) & set(df_assessment_byweek.columns)) 
df_combined=pd.merge(df_vle_byweek, df_assessment_byweek, how='left', on=common_columns)